In [ ]:
# !pip install transformers torch accelerate dotenv pandas openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
import os
model_id = "Qwen/Qwen2.5-0.5B"          # base model, NOT instruct
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
!nvidia-smi

Mon Mar 23 04:55:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from transformers import pipeline

d:\LLM-Learning-Journey\LLM-Learning-Journey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Bart Model for encoder and Decoder architecture

In [2]:
# Load zero-shot classifier (uses BART MNLI model)
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

# Example retail complaints
complaints = [
    "I was charged twice for my purchase and customer care is not responding",
    "The delivery was delayed by 5 days and the package was damaged",
    "The product quality is very poor and not worth the price",
    "I want to return the item but the app is not allowing me to",
    "Payment failed but money got deducted from my account"
]

# Candidate labels (you can customize based on business)
labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue",
    "customer service"
]

# Run classification
for complaint in complaints:
    result = classifier(complaint, candidate_labels=labels)
    
    print(f"\nComplaint: {complaint}")
    print(f"Predicted Label: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.4f}")

d:\LLM-Learning-Journey\LLM-Learning-Journey\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\srini\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 3029.92it/s]



Complaint: I was charged twice for my purchase and customer care is not responding
Predicted Label: payment issue
Confidence: 0.4006

Complaint: The delivery was delayed by 5 days and the package was damaged
Predicted Label: delivery issue
Confidence: 0.9406

Complaint: The product quality is very poor and not worth the price
Predicted Label: product quality
Confidence: 0.9279

Complaint: I want to return the item but the app is not allowing me to
Predicted Label: return/refund
Confidence: 0.5672

Complaint: Payment failed but money got deducted from my account
Predicted Label: payment issue
Confidence: 0.7093


## BERT Model

In [3]:
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

complaints = [
    "I was charged twice for my purchase",
    "Delivery was delayed and package damaged",
    "The product quality is very poor",
    "I want to return the item",
    "Payment failed but money deducted"
]

labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue"
]

for complaint in complaints:
    result = classifier(complaint, candidate_labels=labels)
    
    print(f"\nComplaint: {complaint}")
    print(f"Predicted: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.4f}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2478.30it/s]



Complaint: I was charged twice for my purchase
Predicted: payment issue
Confidence: 0.3073

Complaint: Delivery was delayed and package damaged
Predicted: delivery issue
Confidence: 0.9924

Complaint: The product quality is very poor
Predicted: product quality
Confidence: 0.9875

Complaint: I want to return the item
Predicted: payment issue
Confidence: 0.3106

Complaint: Payment failed but money deducted
Predicted: payment issue
Confidence: 0.9852


## Decoder Only Model (Qwen model)

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
import os
model_id = "Qwen/Qwen2.5-0.5B"          # base model, NOT instruct
device = "cuda" if torch.cuda.is_available() else "cpu"

load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv("hugging_face_token")
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32, token = hf_token).to(device)
model.eval()

complaints = [
    "I was charged twice for my purchase",
    "Delivery was delayed and package damaged",
    "The product quality is very poor",
]

candidate_labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue",
]

def score_label(complaint, label):
    """
    Score how likely the label is to follow the complaint.
    Lower loss = higher probability = better match.
    """
    prompt = f"Complaint: {complaint}\nCategory: {label}"
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return -outputs.loss.item()          # higher score = better fit

for i, complaint in enumerate(complaints, 1):
    print(f"\nComplaint {i}: {complaint}")

    scores = {label: score_label(complaint, label) for label in candidate_labels}
    prediction = max(scores, key=scores.get)

    print(f"Prediction : {prediction}")
    print(f"Scores     :")
    for label, score in sorted(scores.items(), key=lambda x: -x[1]):
        bar = "#" * int((score + 10) * 3)
        print(f"  {label:<20} {score:+.4f}  {bar}")

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 81.55it/s] 



Complaint 1: I was charged twice for my purchase
Prediction : product quality
Scores     :
  product quality      -4.4091  ################
  return/refund        -4.4608  ################
  billing issue        -4.5242  ################
  payment issue        -4.5532  ################
  delivery issue       -4.7067  ###############

Complaint 2: Delivery was delayed and package damaged
Prediction : delivery issue
Scores     :
  delivery issue       -4.1383  #################
  return/refund        -4.1828  #################
  product quality      -4.1881  #################
  billing issue        -4.4514  ################
  payment issue        -4.4758  ################

Complaint 3: The product quality is very poor
Prediction : product quality
Scores     :
  product quality      -3.3380  ###################
  return/refund        -3.8233  ##################
  delivery issue       -3.8726  ##################
  payment issue        -4.0218  #################
  billing issue        -4.0

## Comparison of all 3 models

outputs.loss as scoreResearch technique
Using the model's cross-entropy loss as a proxy for label likelihood is a legitimate zero-shot classification method from research. Lower loss = model finds this completion more probable = higher confidence this is the correct label.

-outputs.loss.item()Correct sign
You negated the loss correctly. Loss is positive (lower = better). By negating, higher score = better match — consistent with BART/DistilBERT confidence scores. Small detail, completely correct.

torch.no_grad()Essential
Disables gradient tracking during inference. Without this, PyTorch builds a computation graph for every forward pass — wastes memory and slows down inference significantly. Always use with torch.no_grad(): for inference. You knew this on Day 4.

qwen_model.eval()Correct
Switches model to evaluation mode — disables dropout layers and batch normalisation behaviour that should only be active during training. eval() + no_grad() together are the standard inference setup. Both used correctly.

torch.softmax(..., dim=0)Correct
Converting raw loss scores to comparable probabilities using softmax. dim=0 is correct here since your scores tensor is 1D (one score per label). This makes Qwen confidence scores comparable to BART/DistilBERT probabilities.

pandas DataFrame outputProduction thinking
Using pandas to structure and display results shows you are thinking about how to present experimental findings, not just print raw output. This is the difference between a notebook experiment and a reproducible comparison study.

In [5]:
import torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
# ── Complaints & labels ────────────────────────────────────────────────────────
complaints = [
    "I was charged twice for my purchase and customer care is not responding",
    "The delivery was delayed by 5 days and the package was damaged",
    "The product quality is very poor and not worth the price",
    "I want to return the item but the app is not allowing me to",
    "Payment failed but money got deducted from my account"
]

labels = [
    "billing issue",
    "delivery issue",
    "product quality",
    "return/refund",
    "payment issue",
    "customer service"
]
hf_token = os.getenv("hugging_face_token")
# ── Model 1: BART MNLI ─────────────────────────────────────────────────────────
print("Loading BART MNLI...")
bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", token = hf_token)

# ── Model 2: DistilBERT MNLI ───────────────────────────────────────────────────
print("Loading DistilBERT MNLI...")
distilbert = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", token = hf_token)

# ── Model 3: Qwen 0.5B Base (likelihood scoring) ──────────────────────────────
print("Loading Qwen2.5-0.5B...")
model_id = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)
qwen_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32, token = hf_token).to(device)
qwen_model.eval()

def qwen_predict(complaint):
    scores = {}
    for label in labels:
        prompt = f"Complaint: {complaint}\nCategory: {label}"
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = qwen_model(**inputs, labels=inputs["input_ids"])
        scores[label] = -outputs.loss.item()
    best_label = max(scores, key=scores.get)
    # softmax for a comparable confidence score
    score_vals = torch.tensor(list(scores.values()))
    confidences = torch.softmax(score_vals, dim=0)
    best_conf = confidences[list(scores.keys()).index(best_label)].item()
    return best_label, best_conf

# ── Run all three & collect results ───────────────────────────────────────────
print("\nRunning classification...\n")
rows = []

for complaint in complaints:
    bart_result      = bart(complaint, candidate_labels=labels)
    distilbert_result = distilbert(complaint, candidate_labels=labels)
    qwen_label, qwen_conf = qwen_predict(complaint)

    rows.append({
        "Complaint"                  : complaint,
        "BART label"                 : bart_result["labels"][0],
        "BART conf"                  : round(bart_result["scores"][0], 4),
        "DistilBERT label"           : distilbert_result["labels"][0],
        "DistilBERT conf"            : round(distilbert_result["scores"][0], 4),
        "Qwen (decoder) label"       : qwen_label,
        "Qwen (decoder) conf"        : round(qwen_conf, 4),
    })

# ── Display ────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_excel("zero_shot_complaint_classification.xlsx", index=False)

Loading BART MNLI...


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7091.37it/s]


Loading DistilBERT MNLI...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6545.34it/s]


Loading Qwen2.5-0.5B...


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 273.80it/s]



Running classification...

                                                              Complaint      BART label  BART conf DistilBERT label  DistilBERT conf Qwen (decoder) label  Qwen (decoder) conf
I was charged twice for my purchase and customer care is not responding   payment issue     0.4006 customer service           0.2591     customer service               0.2161
         The delivery was delayed by 5 days and the package was damaged  delivery issue     0.9406   delivery issue           0.9583     customer service               0.1989
               The product quality is very poor and not worth the price product quality     0.9279  product quality           0.9947      product quality               0.2248
            I want to return the item but the app is not allowing me to   return/refund     0.5672    payment issue           0.2862     customer service               0.2040
                  Payment failed but money got deducted from my account   payment issue     0.709